# Bayesian inference: practice notebook

3 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("b1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
from scipy import stats
_g = np.random.default_rng(11)
Xb = _g.normal(size=(60, 8))
yb = Xb @ np.array([1.5, -2, 0, 0, 1, 0, 0, 0.5]) + _g.normal(0, 1.0, 60)

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJiMSI6IHsic29sdXRpb24iOiAiZGVmIHBvc3RlcmlvcihhLCBiLCBrLCBuKTpcbiAgICByZXR1cm4gYSArIGssIGIgKyBuIC0ga1xuXG5kZWYgY3JlZGlibGUoYSwgYiwgbGV2ZWw9MC45NSk6XG4gICAgdCA9ICgxIC0gbGV2ZWwpIC8gMlxuICAgIHJldHVybiBmbG9hdChzdGF0cy5iZXRhLnBwZih0LCBhLCBiKSksIGZsb2F0KHN0YXRzLmJldGEucHBmKDEgLSB0LCBhLCBiKSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoKTpcbiAgICBhc3NlcnQgdHVwbGUoVVtcInBvc3RlcmlvclwiXSgyLCAzLCA3LCAyMCkpID09ICg5LCAxNiksIFwicG9zdGVyaW9yKDIsIDMsIDcsIDIwKSBzaG91bGQgYmUgKDksIDE2KVwiXG4gICAgZm9yIGFyZ3MgaW4gWyg5LCAxNiwgMC45NSksICgyLCA1MCwgMC45KSwgKDMwLjUsIDQuNSwgMC45OSldOlxuICAgICAgICBhLCBiID0gVVtcImNyZWRpYmxlXCJdKCphcmdzKSwgUltcImNyZWRpYmxlXCJdKCphcmdzKVxuICAgICAgICBjbG9zZShhWzBdLCBiWzBdLCAxZS04LCBmXCJsb3cge2FyZ3N9XCIpOyBjbG9zZShhWzFdLCBiWzFdLCAxZS04LCBmXCJoaWdoIHthcmdzfVwiKVxuVEVTVFMgPSBbKFwicG9zdGVyaW9yIGFuZCBpbnRlcnZhbFwiLCBfdCldXG4iLCAibGFuZyI6ICJweSJ9LCAiYjIiOiB7InNvbHV0aW9uIjogImRlZiBwcm9iX2JfYmVhdHNfYSh4YSwgbmEsIHhiLCBuYiwgZHJhd3M9MjAwXzAwMCwgc2VlZD0wKTpcbiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZClcbiAgICBwYSA9IHJuZy5iZXRhKDEgKyB4YSwgMSArIG5hIC0geGEsIGRyYXdzKVxuICAgIHBiID0gcm5nLmJldGEoMSArIHhiLCAxICsgbmIgLSB4YiwgZHJhd3MpXG4gICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4ocGIgPiBwYSkpLCBmbG9hdChucC5tZWFuKChwYiAtIHBhKSAvIHBhKSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoKmFyZ3MpOlxuICAgIGEgPSBVW1wicHJvYl9iX2JlYXRzX2FcIl0oKmFyZ3MsIGRyYXdzPTQwMF8wMDAsIHNlZWQ9OSk7IGIgPSBSW1wicHJvYl9iX2JlYXRzX2FcIl0oKmFyZ3MsIGRyYXdzPTQwMF8wMDAsIHNlZWQ9MSlcbiAgICBjbG9zZShhWzBdLCBiWzBdLCAwLjAwNiwgXCJQKEIgPiBBKVwiKTsgY2xvc2UoYVsxXSwgYlsxXSwgMC4wMSwgXCJleHBlY3RlZCBsaWZ0XCIpXG5URVNUUyA9IFsoXCI1MDAvNTAwMCB2cyA1NjAvNTAwMFwiLCBsYW1iZGE6IF90KDUwMCwgNTAwMCwgNTYwLCA1MDAwKSksIChcImhpZGRlbjogc21hbGwgc2FtcGxlc1wiLCBsYW1iZGE6IF90KDQsIDQwLCA5LCA0NSkpXVxuIiwgImxhbmciOiAicHkifSwgImIzIjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBSaWRnZVxuXG5kZWYgbWFwX2dhdXNzaWFuKFgsIHksIHNpZ21hMiwgdGF1Mik6XG4gICAgbGFtID0gc2lnbWEyIC8gdGF1MlxuICAgIHJldHVybiBucC5saW5hbGcuc29sdmUoWC5UIEAgWCArIGxhbSAqIG5wLmV5ZShYLnNoYXBlWzFdKSwgWC5UIEAgeSlcblxuZGVmIHJpZGdlX3NrbGVhcm4oWCwgeSwgc2lnbWEyLCB0YXUyKTpcbiAgICByZXR1cm4gUmlkZ2UoYWxwaGE9c2lnbWEyIC8gdGF1MiwgZml0X2ludGVyY2VwdD1GYWxzZSkuZml0KFgsIHkpLmNvZWZfXG4iLCAidGVzdHMiOiAiXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5kZWYgc2FtZV9rZXlzKGEsIGIsIHdoYXQ9XCJyZXN1bHRcIik6XG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgZGljdCksIGZcInt3aGF0fSBzaG91bGQgYmUgYSBkaWN0LCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgc2V0KGEpID09IHNldChiKSwgZlwie3doYXR9IGtleXMge3NvcnRlZChhKX0sIGV4cGVjdGVkIHtzb3J0ZWQoYil9XCJcblxuZGVmIF90KHMyLCB0Mik6XG4gICAgciA9IFJbXCJtYXBfZ2F1c3NpYW5cIl0oWGIsIHliLCBzMiwgdDIpXG4gICAgZm9yIG5hbWUgaW4gKFwibWFwX2dhdXNzaWFuXCIsIFwicmlkZ2Vfc2tsZWFyblwiKTpcbiAgICAgICAgYSA9IG5wLmFzYXJyYXkoVVtuYW1lXShYYiwgeWIsIHMyLCB0MikpXG4gICAgICAgIGFzc2VydCBhLnNoYXBlID09IHIuc2hhcGUsIGZcIntuYW1lfTogc2hhcGUge2Euc2hhcGV9LCBleHBlY3RlZCB7ci5zaGFwZX1cIlxuICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4ocikpOiBjbG9zZShhW2ldLCByW2ldLCAxZS02LCBmXCJ7bmFtZX0gXHUwM2IyW3tpfV1cIilcblRFU1RTID0gWyhcIlx1MDNjM1x1MDBiMiA9IDEsIFx1MDNjNFx1MDBiMiA9IDAuNVwiLCBsYW1iZGE6IF90KDEuMCwgMC41KSksIChcImhpZGRlbjogc3Ryb25nIHByaW9yXCIsIGxhbWJkYTogX3QoMi4wLCAwLjAxKSldXG4iLCAibGFuZyI6ICJweSJ9fQ==").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkgaW1wb3J0IHN0YXRzCl9nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDExKQpYYiA9IF9nLm5vcm1hbChzaXplPSg2MCwgOCkpCnliID0gWGIgQCBucC5hcnJheShbMS41LCAtMiwgMCwgMCwgMSwgMCwgMCwgMC41XSkgKyBfZy5ub3JtYWwoMCwgMS4wLCA2MCk=").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Beta-Binomial update and a 95% credible interval
*Easy · id `b1`*

<p>A conversion rate has prior Beta(a, b). You observe k conversions in n trials.</p>
<p>Write <code>posterior(a, b, k, n)</code> returning the posterior parameters, and <code>credible(a, b, level=0.95)</code> returning the equal-tailed credible interval with <code>stats.beta.ppf</code>.</p>
<p>Compare the interval for a Beta(1, 1) prior with the frequentist Wald CI. When do they differ most?</p>

**Theory to know cold**
<ul><li>Posterior ∝ p^{k}(1−p)^{n−k} · p^{a−1}(1−p)^{b−1} = Beta(a + k, b + n − k): the prior acts as a − 1 successes and b − 1 failures seen before.</li><li>Posterior mean (a + k)/(a + b + n) is a weighted average of the prior mean and k/n.</li></ul>

In [ ]:
def posterior(a, b, k, n):
    # your code here
    pass

def credible(a, b, level=0.95):
    # your code here
    pass

# Sample run
try:
    a, b = posterior(1, 1, 12, 100); print(a, b, credible(a, b))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("b1")

---
## 2. Bayesian A/B test: probability that treatment beats control
*Medium · id `b2`*

<p>Write <code>prob_b_beats_a(xa, na, xb, nb, draws=200_000, seed=0)</code>: with Beta(1, 1) priors, draw from both posteriors and return P(p_B &gt; p_A). Also return the expected relative lift E[(p_B − p_A)/p_A], as a tuple <code>(prob, lift)</code>.</p>
<p>How does P(B &gt; A) relate to the one-sided p-value of the z-test on the same data?</p>

**Theory to know cold**
<ul><li>With flat priors and large n, P(B &gt; A | data) ≈ 1 − one-sided p-value, but it answers the question stakeholders actually ask.</li><li>Peeking at a Bayesian test still inflates false decisions if you stop as soon as it crosses a threshold.</li></ul>

In [ ]:
def prob_b_beats_a(xa, na, xb, nb, draws=200_000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(prob_b_beats_a(500, 5000, 560, 5000))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("b2")

---
## 3. MAP vs MLE: ridge regression is a Gaussian prior
*Medium · id `b3`*

<p>Model y = Xβ + ε, ε ~ N(0, σ²I), with prior β ~ N(0, τ²I).</p>
<p>On paper: show the MAP estimate minimises ‖y − Xβ‖² + (σ²/τ²)‖β‖². Then write <code>map_gaussian(X, y, sigma2, tau2)</code> with the closed form (XᵀX + λI)⁻¹Xᵀy, λ = σ²/τ² (use <code>np.linalg.solve</code>), and <code>ridge_sklearn(X, y, sigma2, tau2)</code> with <code>Ridge(alpha=λ, fit_intercept=False)</code>. They must agree.</p>
<p>What prior gives the lasso?</p>

**Theory to know cold**
<ul><li>A Laplace (double-exponential) prior gives the lasso; MAP with a flat prior is the MLE, i.e. OLS.</li><li>As τ² → ∞ the prior vanishes and ridge → OLS; as τ² → 0, β̂ → 0.</li></ul>

In [ ]:
from sklearn.linear_model import Ridge

def map_gaussian(X, y, sigma2, tau2):
    # your code here
    pass

def ridge_sklearn(X, y, sigma2, tau2):
    # your code here
    pass

# Sample run
try:
    print(map_gaussian(Xb, yb, 1.0, 0.5).round(3)); print(ridge_sklearn(Xb, yb, 1.0, 0.5).round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("b3")